### README

Identify NAGNAG disease-related variants annottated in AGAIN database.

*Output coordinates at base 1

** Ref: Zhang, Peng, et al. "Genome-wide detection of human intronic AG-gain variants located between splicing branchpoints and canonical splice acceptor sites." Proceedings of the National Academy of Sciences 120.46 (2023): e2314225120.

### Requirements

In [ ]:
# Install
"""
openpyxl==3.1.5
pandas==2.2.3
"""

In [ ]:
# Import libraries
import pandas as pd

### Constants

In [ ]:
# Input: dir with AGAIN variants tables
AGAIN_DIR = '/PATH/TO/DATA' 
# Input: tables to include
AGAIN_LIST = ['pnas.2314225120.sd01_wo1strow.xlsx', 'irf7.xlsx', 'pnas.2314225120.sd04.xlsx'] #sd01 without 1st row from AGAIN paper, IRF7 comes from the text in AGAIN paper, sd04 from AGAIN paper

# Input: reference genome annotation
GTF_PATH = '/PATH/TO/gencode.v44lift37.annotation.gtf.gz'

# Output: set directory
OUTPUT_DIR = '/PATH/TO/OUTPUT'

### Functions

In [ ]:
# LOAD AGAIN VARIANTS FILES INTO A SINGLE DATAFRAME
 
def load_again_files(file_list, base_dir):
    
    cols = ['GENE', 'CHR', 'POS', 'REF', 'ALT', 'STR', 'TYPE', 'AGAIN_ACC_DIST']
    
    dfs = []
    
    for file in file_list:
        path = f"{base_dir}/{file}"
        df = pd.read_excel(path, usecols=cols)
        dfs.append(df)

    again_df = pd.concat(dfs, ignore_index=True)
    
    return again_df


In [ ]:
# EXTRACT INTRONS COORDINATES FROM GTF FILE

def extract_introns(gtf_df: pd.DataFrame) -> pd.DataFrame:

    # Filter only exon entries
    exon_df = gtf_df[gtf_df['Type'] == 'exon'].copy()

    #replace , by ; in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(',',';')
    #replace : by = in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(':','=')

    # Extract parent mRNA (transcript_id)
    exon_df['mRNA'] = (
        exon_df['Attributes']
        .str.split('transcript_id "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Extract parent gene (gene_id)
    exon_df['GENEID'] = (
        exon_df['Attributes']
        .str.split('gene_id "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Sort by gene, transcript, and genomic start
    exon_df = exon_df.sort_values(by=['GENEID', 'mRNA', 'Start'])

    # Assign coordinates of the downstream exon ("below")
    exon_df['Start_Below'] = exon_df.groupby('mRNA')['Start'].shift(-1)
    exon_df['End_Below'] = exon_df.groupby('mRNA')['End'].shift(-1)

    # Drop rows where downstream exon doesn't exist
    exon_df = exon_df.dropna(subset=['Start_Below', 'End_Below']).copy()

    # Calculate intron coordinates
    exon_df['Intron_Start'] = exon_df['End'] + 1
    exon_df['Intron_End'] = exon_df['Start_Below'] - 1

    # Keep relevant columns and remove duplicates
    intron_df = exon_df[['SeqID', 'Intron_Start', 'Intron_End', 'GENEID', 'STR']]
    # rename columns
    intron_df = intron_df.rename(columns={
        'SeqID': 'CHR',
        'Intron_Start': 'START',
        'Intron_End': 'END'
    })
    intron_df = intron_df.drop_duplicates()

    return intron_df

In [ ]:
# GET LONGEST INTRON POSITION (3' END) FROM NAGNAG AGAIN VARIANT

def get_longest_intron(row):
    pos = row['POS']
    ref = row['REF']
    alt = row['ALT']
    strand = row['STR']
    delta = len(alt) - len(ref)


    # Case 1: point mutations
    if delta == 0:
        if strand == '+':
            if alt.upper() == 'A':
                return pos + 4 
            elif alt.upper() == 'G':
                return pos + 3 
            else: # error case
                raise ValueError(f"Unexpected ALT value: {alt} at position {pos}")
        elif strand == '-':
            if alt.upper() == 'T':
                return pos - 4 
            elif alt.upper() == 'C':
                return pos - 3 
            else: # error case
                raise ValueError(f"Unexpected ALT value: {alt} at position {pos}")
    
    # case 2: insertions
    elif delta > 0:
        raise ValueError(f"Unexpected ALT value: {alt} at position {pos}")
        
    # Case 3: deletions
    elif delta < 0:
        raise ValueError(f"Unexpected ALT value: {alt} at position {pos}")

In [ ]:
# GET SHORT AND LONG INTRON COORDINATES BASED ON THE LONGEST INTRON POSITION AT 3' END

def get_nagnag_introns(
        again_df: pd.DataFrame, intron_df: pd.DataFrame, intron_pos_col='Longest_Intron_Pos'
        ) -> pd.DataFrame:

    # Split by strand
    plus_df = again_df[again_df['STR'] == '+']
    minus_df = again_df[again_df['STR'] == '-']

    # '+' strand merge (Pos == Start)
    merged_plus = pd.merge(
        plus_df,
        intron_df,
        left_on=[intron_pos_col, 'CHR', 'STR'],
        right_on=['END', 'CHR', 'STR'],
        how='left').rename(columns={
        'END': 'longIE',
        'START': 'longIS'
    })
    merged_plus['shortIS'] =  merged_plus['longIS']
    merged_plus['shortIE'] =  merged_plus['longIE'] - 3

    # '-' strand merge (Pos == End)
    merged_minus = pd.merge(
        minus_df,
        intron_df,
        left_on=[intron_pos_col, 'CHR', 'STR'],
        right_on=['START', 'CHR', 'STR'],
        how='left'
    ).rename(columns={
        'START': 'longIS',
        'END': 'longIE'
    })
    merged_minus['shortIS'] =  merged_minus['longIS'] + 3
    merged_minus['shortIE'] =  merged_minus['longIE'] 

    # Combine both
    merged_df = pd.concat([merged_plus, merged_minus], ignore_index=True)

    return merged_df

### Analysis

##### 1. Select NAGNAG AGAIN variants

In [ ]:
# load AGAIN variants tables into a single dataframe
again_df = load_again_files(AGAIN_LIST, AGAIN_DIR)

# selects only NAGNAG created by SNVs or insertions
again_df = again_df[(again_df['TYPE']=='snv') | (again_df['TYPE'].str.contains('ins'))]
again_df = again_df[again_df['AGAIN_ACC_DIST'] == -3] # only nagnag

# clean up dataframe
again_df = again_df[['GENE', 'CHR', 'POS', 'REF', 'ALT', 'STR', 'AGAIN_ACC_DIST', 'TYPE']]
again_df = again_df.drop_duplicates()
again_df.reset_index(drop=True, inplace=True)

again_df

##### 2. Get intronic coordinates of the selected NAGNAG AGAIN variants

2.1 Extract all annotated introns coordinates

In [ ]:
# genome annotation
gtf_df = pd.read_table(GTF_PATH,
                       names = ['SeqID', 'Source', 'Type', 'Start', 'End', 'Score', 'STR', 'Phase', 'Attributes'],
                       comment='#',
                       low_memory=False) 


# get intron coordinates from gtf file
intron_df = extract_introns(gtf_df)
intron_df

2.2 Find longest intron position of the selected AGAIN variants

In [ ]:
# get longest intron position only for snv 
mask = again_df['TYPE'] == 'snv'

again_df.loc[mask, 'Longest_Intron_Pos'] = (
    again_df.loc[mask]
    .apply(get_longest_intron, axis=1)
)

again_df


In [ ]:
# mannually add position to not snv
again_df.loc[0, 'Longest_Intron_Pos'] = again_df.loc[0, 'POS'] + 2 #ADCK3 insertion
again_df

2.3 Get intronic coordiantes of the selected AGAIN variants

In [ ]:
# merge with intron dataframe
again_df = get_nagnag_introns(again_df, intron_df, intron_pos_col='Longest_Intron_Pos')


again_df

##### 3. Filter and save AGAIN variants coordinates

In [ ]:
# keep only relevant columns
again_df = again_df[['GENE', 'CHR', 'STR', 
         'shortIS', 'shortIE', 'longIS', 'longIE',
         'POS', 'REF', 'ALT']].copy()

# drop duplicates
again_df = again_df.drop_duplicates()
again_df

In [ ]:
# save the merged dataframe to a csv file
again_df.to_csv(f'{OUTPUT_DIR}/AGAIN_variants.csv', index=False)